In [1]:
import pandas as pd
import numpy as np
import lightkurve as lk
from scipy.interpolate import interp1d
import os
import glob
import concurrent.futures
from tqdm import tqdm 

In [2]:
KOI_FILE = "q1_q8_koi_2025.02.03_04.12.15.csv"
OUTPUT_DIR = "processed_data_new"

# Settings
FIXED_LENGTH = 2000      # Input size for the CNN
SAMPLES_PER_CLASS = 500  # Total samples to get
BATCH_SIZE = 50          # Save every 50 stars
MAX_WORKERS = 4          # Safe number for threads
USE_SINGLE_QUARTER = True # True = Fast (Minutes), False = Slow (Hours)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

In [3]:
def process_one_star(task_data):
    row = task_data['row']
    label_val = task_data['label_val']
    class_name = task_data['class_name']
    
    kic = int(row['kepid'])
    search_id = f"KIC {kic}"
    
    try:
        # Search
        search = lk.search_lightcurve(search_id, mission='Kepler', author='Kepler', cadence='long')
        if len(search) == 0: return None
            
        # Download
        if USE_SINGLE_QUARTER:
            lc_collection = search[0].download()
            raw_lc = lc_collection.normalize() 
        else:
            lc_collection = search.download_all()
            if lc_collection is None: return None
            raw_lc = lc_collection.stitch()
        
        # Clean & Flatten
        clean_lc = raw_lc.remove_nans().flatten(window_length=901).remove_outliers(sigma=5)
        
        # Interpolate
        flux = clean_lc.flux.value
        f_interp = interp1d(np.linspace(0, 1, len(flux)), flux, kind='linear')
        flux_fixed = f_interp(np.linspace(0, 1, FIXED_LENGTH))
        
        return {
            'flux': flux_fixed,
            'label': label_val,
            'meta': {
                'kic': kic, 'label': class_name,
                'period': row['koi_period'], 'duration': row['koi_duration']
            }
        }
    except:
        return None

In [4]:
def run_smart_batch(dataframe, label_val, class_name):
    print(f"\n--- Preparing {class_name} ---")
    
    # Check for existing checkpoints
    existing_files = glob.glob(f"{OUTPUT_DIR}/temp_{class_name}_*_X.npy")
    skip_count = 0
    
    if existing_files:
        # Find the highest batch number saved
        batches = [int(f.split('_')[-2]) for f in existing_files]
        max_batch = max(batches)
        # We skip everything up to that batch
        # (Batch 0 is items 0-49, Batch 1 is 50-99...)
        # So if Batch 1 exists, we have processed (1+1)*50 = 100 items.
        skip_count = (max_batch + 1) * BATCH_SIZE
        print(f"  -> Found {len(existing_files)} checkpoint files.")
        print(f"  -> Resuming from sample #{skip_count}")
    
    # Slice the dataframe to only process NEW stars
    if skip_count >= len(dataframe):
        print("  -> All data already processed! Skipping download.")
        return
    
    remaining_df = dataframe.iloc[skip_count:]
    print(f"  -> Processing {len(remaining_df)} new stars...")
    
    # Prepare Tasks
    tasks = []
    for _, row in remaining_df.iterrows():
        tasks.append({'row': row, 'label_val': label_val, 'class_name': class_name})
        
    collected_X = []
    collected_y = []
    collected_meta = []
    
    # Run Workers
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_one_star, t): t for t in tasks}
        
        # We start the batch_counter from where we left off
        current_batch_idx = (skip_count // BATCH_SIZE) 
        
        for i, future in tqdm(enumerate(concurrent.futures.as_completed(futures)), total=len(tasks)):
            result = future.result()
            
            if result is not None:
                collected_X.append(result['flux'])
                collected_y.append(result['label'])
                collected_meta.append(result['meta'])
            
            # Save Checkpoint
            # Logic: If we have collected BATCH_SIZE new items, save them
            if len(collected_X) > 0 and len(collected_X) % BATCH_SIZE == 0:
                # IMPORTANT: Use a NEW batch number so we don't overwrite old ones
                save_idx = current_batch_idx + (len(collected_X) // BATCH_SIZE)
                
                # Save chunk
                # We only save the LAST BATCH_SIZE items to avoid duplicates in memory
                chunk_X = collected_X[-BATCH_SIZE:]
                chunk_y = collected_y[-BATCH_SIZE:]
                chunk_meta = collected_meta[-BATCH_SIZE:]
                
                np.save(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_X.npy", np.array(chunk_X))
                np.save(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_y.npy", np.array(chunk_y))
                pd.DataFrame(chunk_meta).to_csv(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_meta.csv", index=False)

    # Save any remaining leftovers (tail end)
    if len(collected_X) % BATCH_SIZE != 0:
        save_idx = current_batch_idx + (len(collected_X) // BATCH_SIZE) + 1
        tail_idx = -(len(collected_X) % BATCH_SIZE)
        
        chunk_X = collected_X[tail_idx:]
        chunk_y = collected_y[tail_idx:]
        chunk_meta = collected_meta[tail_idx:]
        
        np.save(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_X.npy", np.array(chunk_X))
        np.save(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_y.npy", np.array(chunk_y))
        pd.DataFrame(chunk_meta).to_csv(f"{OUTPUT_DIR}/temp_{class_name}_{save_idx}_meta.csv", index=False)


In [5]:
def force_merge_all():
    print("\n--- Finalizing & Merging Dataset ---")
    
    all_X = []
    all_y = []
    all_meta = []
    
    # 1. Scan for ALL temp files
    # This grabs Confirmed AND False Positives automatically
    x_files = sorted(glob.glob(f"{OUTPUT_DIR}/temp_*_X.npy"))
    
    if not x_files:
        print("CRITICAL ERROR: No temp files found. Nothing to merge.")
        return

    print(f"Found {len(x_files)} data chunks. Merging...")
    
    for x_f in tqdm(x_files):
        base = x_f.replace("_X.npy", "")
        y_f = base + "_y.npy"
        m_f = base + "_meta.csv"
        
        if os.path.exists(y_f) and os.path.exists(m_f):
            all_X.append(np.load(x_f))
            all_y.append(np.load(y_f))
            all_meta.append(pd.read_csv(m_f))
            
    # 2. Concatenate
    final_X = np.concatenate(all_X)
    final_y = np.concatenate(all_y)
    final_meta = pd.concat(all_meta, ignore_index=True)
    
    # 3. Save Masters
    np.save(f"{OUTPUT_DIR}/X_data.npy", final_X)
    np.save(f"{OUTPUT_DIR}/y_labels.npy", final_y)
    final_meta.to_csv(f"{OUTPUT_DIR}/dataset_key.csv", index=False)
    
    print(f"\nDONE! Pipeline Complete.")
    print(f"Final Dataset Size: {len(final_X)} stars")
    print(f"Saved to folder: {OUTPUT_DIR}/")

# MAIN EXECUTION BLOCK
if __name__ == "__main__":
    print("--- STARTING ROBUST EXOPLANET PIPELINE ---")
    
    # 1. Load List
    try:
        df = pd.read_csv(KOI_FILE, comment='#')
    except:
        df = pd.read_csv(KOI_FILE)
        
    pos = df[df['koi_disposition'] == 'CONFIRMED'].sample(frac=1, random_state=42).head(SAMPLES_PER_CLASS)
    neg = df[df['koi_disposition'] == 'FALSE POSITIVE'].sample(frac=1, random_state=42).head(SAMPLES_PER_CLASS)
    
    # 2. Run False Positives (Smart Resume)
    run_smart_batch(neg, 0, "FALSE_POS")
    
    # 3. Run Confirmed Planets (Smart Resume)
    run_smart_batch(pos, 1, "CONFIRMED")
    
    # 4. Merge Everything
    force_merge_all()

--- STARTING ROBUST EXOPLANET PIPELINE ---

--- Preparing FALSE_POS ---
  -> Found 10 checkpoint files.
  -> Resuming from sample #500
  -> All data already processed! Skipping download.

--- Preparing CONFIRMED ---
  -> Found 10 checkpoint files.
  -> Resuming from sample #500
  -> All data already processed! Skipping download.

--- Finalizing & Merging Dataset ---
Found 20 data chunks. Merging...


100%|██████████| 20/20 [00:00<00:00, 26.39it/s]



DONE! Pipeline Complete.
Final Dataset Size: 3974 stars
Saved to folder: processed_data_new/
